In [ ]:
# 安裝 LINE Bot、Flask（伺服器）、ngrok（公開網址）、requests（API）
!pip install Flask pyngrok line-bot-sdk requests --quiet

# 安裝 Gemini AI SDK
!pip install google-genai --quiet

In [ ]:
# 從 Colab 安全儲存區讀取 API / Token
from google.colab import userdata

ngrok_authtoken = "3ClDmzH22sqWEbRSdze4yo4Zgpf_7ycL9z7EcZyfCYuL92PAR"
line_channel_access_token = "1OQmRLQRLLrqhossKfNHRNPO7oKJBICehSGz0ZmTf/I3JuDjHP2cYZX9L8u/MRrHuO6MmNe+szkjDQSHLoKreAvJxy/k7n0QhuWs/M31wA6meIcAA7nMmcPUsi9VR2SWtKdL3ny/TSnTrMKfBml3CwdB04t89/1O/w1cDnyilFU="
line_channel_secret = "79de232aeab6cf1e0df8c7723791c4bc"
gemini_api_key = "AQ.Ab8RN6Kuq6rls6xKpuhMYi31J6Bsm5W8byezu3CdODPts_n-dA"

port = 5051  # Flask 埠號

In [ ]:
# ngrok：把本機 Flask 變成公開網址
from pyngrok import ngrok
import requests

In [ ]:
# 避免「同一個網址已經在使用」錯誤
ngrok.kill()

In [ ]:
import requests
from pyngrok import ngrok

# 🔥 一定要先關掉舊的 ngrok（避免 ERR_NGROK_334）
ngrok.kill()

# 設定 token
ngrok.set_auth_token(ngrok_authtoken)

# 開新的 tunnel（⚠️ 不要 name）
tunnel = ngrok.connect(5051)

# 取得網址
webhook_url = tunnel.public_url
print("Webhook URL:", webhook_url)

# 更新 LINE webhook
def update_line_webhook(webhook_url):

    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"

    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }

    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    print("status:", response.status_code)
    print("response:", response.text)

update_line_webhook(webhook_url)

In [ ]:
from google import genai
from google.genai.types import GenerateContentConfig

# 建立 AI client
client = genai.Client(api_key=gemini_api_key)

# 建立「有記憶對話」
chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        response_modalities=["TEXT"]
    )
)

In [ ]:
# 把文字丟給 AI，並保留上下文記憶
def stateful_query(payload):

    response = chat.send_message(message=payload)

    return response.text

In [ ]:
result = stateful_query("簡介明新科技大學")
print(result)

In [ ]:
result2 = stateful_query("校長是誰？")
print(result2)

In [ ]:
from flask import Flask, request, abort

# LINE webhook 驗證
from linebot.v3 import WebhookHandler
from linebot.v3.exceptions import InvalidSignatureError

# LINE 回覆 API
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)

# LINE webhook event
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

# 建立 Flask 伺服器
app = Flask(__name__)

# LINE API 設定
configuration = Configuration(
    access_token=line_channel_access_token
)

handler = WebhookHandler(line_channel_secret)

# LINE 收到訊息的入口
@app.route("/", methods=["POST"])
def callback():

    signature = request.headers["X-Line-Signature"]
    body = request.get_data(as_text=True)

    print("BODY:", body)

    try:
        handler.handle(body, signature)

    except InvalidSignatureError:
        abort(400)

    return "OK"

# 當使用者傳文字訊息時觸發
@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):

    text = event.message.text

    with ApiClient(configuration) as api_client:

        line_bot_api = MessagingApi(api_client)

        # 如果開頭是 AI → 丟給 Gemini
        if text.startswith("AI "):

            prompt = text[3:]
            reply_text = stateful_query(prompt)

            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        # 一般訊息 → 原封不動回覆（測試用）
        else:

            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[
                        TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)
                    ]
                )
            )

# 啟動伺服器（一定要公開 0.0.0.0）
app.run(
    host="0.0.0.0",
    port=port
)